<a href="https://colab.research.google.com/github/Davron030901/Numpy/blob/main/ERROR_HANDLING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```
np.errstate() - error handling
np.seterr() - global error settings
np.isnan() / np.isinf() - invalid values ​​checking
try-except - Python exception handling
np.nan_to_num() - invalid values ​​correction

```
ERROR TYPES:

  divide: Division by zero
  invalid: Invalid operations (sqrt(-1))
  over: Overflow
  under: Underflow

ERROR ACTIONS:

  'ignore': No warning
  'warn': Show warning (default)
  'raise': Raise exception
  'print': Print to stdout

In [25]:
import numpy as np

# np.errstate() Context Manager! 🛡️

In [26]:
arr = np.array([1, 2, 0, 4, -1])
print("Array:", arr)

Array: [ 1  2  0  4 -1]


WITHOUT error control (warnings appear)

In [27]:
print("Method 1: No error handling")
result1 = 1/arr
result2 = np.sqrt(arr)
print("Division result:", result1)
print("Sqrt result:", result2)

Method 1: No error handling
Division result: [ 1.    0.5    inf  0.25 -1.  ]
Sqrt result: [1.         1.41421356 0.         2.                nan]


/tmp/ipython-input-254/571179895.py:2: RuntimeWarning: divide by zero encountered in divide
  result1 = 1/arr
/tmp/ipython-input-254/571179895.py:3: RuntimeWarning: invalid value encountered in sqrt
  result2 = np.sqrt(arr)


WITH np.errstate() (suppress warnings)

In [28]:
print("Method 2: Using np.errstate()")
with np.errstate(divide='ignore', invalid='ignore'):
  result1 = 1/arr
  result2 = np.sqrt(arr)
print("Division result:", result1)
print("Sqrt result:", result2)

Method 2: Using np.errstate()
Division result: [ 1.    0.5    inf  0.25 -1.  ]
Sqrt result: [1.         1.41421356 0.         2.                nan]


Check for invalid values

In [29]:
print("Has inf:", np.isinf(result1).any())
print("Has nan:", np.isnan(result2).any())

Has inf: True
Has nan: True


# Detecting Invalid Values! 🔍

In [30]:
arr = np.array([1, 2, 0, -1, 4])

with np.errstate(divide='ignore', invalid='ignore'):
  division = 1 / arr
  sqrt_vals = np.sqrt(arr)

print("Original:", arr)
print("Division result:", division)
print("Sqrt result:", sqrt_vals)

Original: [ 1  2  0 -1  4]
Division result: [ 1.    0.5    inf -1.    0.25]
Sqrt result: [1.         1.41421356 0.                nan 2.        ]


 CHECK for inf

In [31]:
inf_mask = np.isinf(division)
print("Has inf:", inf_mask)
print("Inf values:", division[inf_mask])
print(f"Inf count: {inf_mask.sum()}")

Has inf: [False False  True False False]
Inf values: [inf]
Inf count: 1


CHECK for nan

In [32]:
nan_mask = np.isnan(sqrt_vals)
print("Has nan:", nan_mask)
print("Nan values:", sqrt_vals[nan_mask])
print(f"Nan count: {nan_mask.sum()}")

Has nan: [False False False  True False]
Nan values: [nan]
Nan count: 1


CHECK for finite (not inf, not nan)

In [33]:
finite_mask = np.isfinite(division)
print("Has finite:", finite_mask)
print("Finite values:", division[finite_mask])

Has finite: [ True  True False  True  True]
Finite values: [ 1.    0.5  -1.    0.25]


#Fixing Invalid Values! 🔧

In [34]:
arr = np.array([1.0, 2.0, 0.0, -1.0, 4.0])

with np.errstate(divide='ignore', invalid='ignore'):
    result = 1 / arr
    sqrt_result = np.sqrt(arr)

print("Original division:", result)
print("Original sqrt:", sqrt_result)

Original division: [ 1.    0.5    inf -1.    0.25]
Original sqrt: [1.         1.41421356 0.                nan 2.        ]


METHOD 1: Replace manually

In [35]:
fixed1 = result.copy()
fixed1[np.isinf(fixed1)] = 0
print("Fixed division (inf→0):", fixed1)

Fixed division (inf→0): [ 1.    0.5   0.   -1.    0.25]


In [36]:
fixed2 = sqrt_result.copy()
fixed2[np.isnan(fixed2)] = 0
print("Fixed sqrt (nan→0):", fixed2)

Fixed sqrt (nan→0): [1.         1.41421356 0.         0.         2.        ]


METHOD 2: np.nan_to_num()

Replaces nan→0, inf→large number, -inf→small number

In [37]:
fixed_auto = np.nan_to_num(result)
print("Auto-fixed division:", fixed_auto)

Auto-fixed division: [ 1.00000000e+000  5.00000000e-001  1.79769313e+308 -1.00000000e+000
  2.50000000e-001]


In [38]:
fixed_auto2 = np.nan_to_num(sqrt_result)
print("Auto-fixed sqrt:", fixed_auto2)

Auto-fixed sqrt: [1.         1.41421356 0.         0.         2.        ]


METHOD 3: Custom replacement

In [39]:
fixed_custom = np.nan_to_num(result,
                             nan=0.0,
                             posinf=999.0,
                             neginf=-999.0)
print("Custom-fixed division:", fixed_custom)

Custom-fixed division: [ 1.00e+00  5.00e-01  9.99e+02 -1.00e+00  2.50e-01]


#Try-Except with NumPy! 🎯

In [40]:
arr = np.array([1, 2, 3, 4, 5])

print("Test 1: Index error")
try:
  value = arr[100]
  print(f"Value: {value}")
except IndexError as e:
  print(f"IndexError: {e}")
  print("Using safe default: 0")
  value=0

Test 1: Index error
IndexError: index 100 is out of bounds for axis 0 with size 5
Using safe default: 0


SHAPE ERROR

In [41]:
print("Test 2: Shape error")
matrix = np.array([[1, 2], [3, 4]])
vector = np.array([1, 2, 3])

Test 2: Shape error


In [42]:
try:
  result = matrix + vector
except ValueError as e:
  print(f"ValuError: {e}")
  print("Shapes incompatible, using broadcast-compatible vector")
  vector = np.array([1, 2])
  result = matrix + vector
  print("Result:", result)

ValuError: operands could not be broadcast together with shapes (2,2) (3,) 
Shapes incompatible, using broadcast-compatible vector
Result: [[2 4]
 [4 6]]


#DTYPE ERROR

In [43]:
print("Test 3: Dtype error")

try:
  arr_float = np.array([1.5, 2.7, 3.9])
  arr_int = arr_float.astype(np.int8, casting='safe')
except TypeError as e:
  print(f"TypeError: {e}")
  print("Using safe casting")
  arr_int = arr_float.astype(np.int8, casting='unsafe')
  print("Result:", arr_int)

Test 3: Dtype error
TypeError: Cannot cast array data from dtype('float64') to dtype('int8') according to the rule 'safe'
Using safe casting
Result: [1 2 3]


#Safe Division

In [44]:
arr = np.array([10, 5, 0, 2, 0, 8])
devision = 100 / arr

print("Original array:", arr)
print("Devision:", devision)

Original array: [10  5  0  2  0  8]
Devision: [10.  20.   inf 50.   inf 12.5]


/tmp/ipython-input-254/2515735588.py:2: RuntimeWarning: divide by zero encountered in divide
  devision = 100 / arr


In [45]:
perform_division = np.errstate(divide='ignore', invalid='ignore')
with perform_division:
  devision = 100 / arr
  print("Safe devision:", devision)

Safe devision: [10.  20.   inf 50.   inf 12.5]


In [46]:
inf_value = np.isinf(devision)
print("Has inf:", inf_value)

Has inf: [False False  True False  True False]


In [47]:
replace_inf0 = np.where(inf_value, 0, devision)
print("Replace inf with 0:", replace_inf0)

Replace inf with 0: [10.  20.   0.  50.   0.  12.5]


In [48]:
print("Original:", arr)
print("Before:", devision)
print("After:", replace_inf0)

Original: [10  5  0  2  0  8]
Before: [10.  20.   inf 50.   inf 12.5]
After: [10.  20.   0.  50.   0.  12.5]


In [49]:
count_replace = inf_value.sum()
print("Count of inf:", count_replace)

Count of inf: 2


#Safe Square Root

In [50]:
arr = np.array([4, 9, -1, 16, -4, 25, 0])

print("Original array:", arr)

Original array: [ 4  9 -1 16 -4 25  0]


In [51]:
cal_sqrt = np.sqrt(arr)
print("Calculated sqrt:", cal_sqrt)

Calculated sqrt: [ 2.  3. nan  4. nan  5.  0.]


/tmp/ipython-input-254/2535965494.py:1: RuntimeWarning: invalid value encountered in sqrt
  cal_sqrt = np.sqrt(arr)


In [53]:
with_ereor = np.errstate(invalid='ignore')
with with_ereor:
  cal_sqrt = np.sqrt(arr)
  print("Safe sqrt:", cal_sqrt)

Safe sqrt: [ 2.  3. nan  4. nan  5.  0.]


In [54]:
find_nan = np.isnan(cal_sqrt)
print("Has nan:", find_nan)

Has nan: [False False  True False  True False False]


In [55]:
replace_nan = np.nan_to_num(cal_sqrt, nan=0.0)
print("Replace nan with 0:", replace_nan)

Replace nan with 0: [2. 3. 0. 4. 0. 5. 0.]


In [56]:
print("Original:", arr)
print("Before:", cal_sqrt)
print("After:", replace_nan)

Original: [ 4  9 -1 16 -4 25  0]
Before: [ 2.  3. nan  4. nan  5.  0.]
After: [2. 3. 0. 4. 0. 5. 0.]


In [57]:
mean_of_valid = replace_nan[~np.isnan(replace_nan)].mean()
print("Mean of valid values:", mean_of_valid)

Mean of valid values: 2.0


#Robust Data Processing

In [58]:
data =np.array([1.5, 2.0, 0.0, -1.0, 3.5, 0.0, 4.0])

print("Original:", data)

Original: [ 1.5  2.   0.  -1.   3.5  0.   4. ]


In [62]:
log = np.log(data)
print("Log:", log)

division = 1 / data
print("Division:", division)

Log: [0.40546511 0.69314718       -inf        nan 1.25276297       -inf
 1.38629436]
Division: [ 0.66666667  0.5                inf -1.          0.28571429         inf
  0.25      ]


/tmp/ipython-input-254/3903399755.py:1: RuntimeWarning: divide by zero encountered in log
  log = np.log(data)
/tmp/ipython-input-254/3903399755.py:1: RuntimeWarning: invalid value encountered in log
  log = np.log(data)
/tmp/ipython-input-254/3903399755.py:4: RuntimeWarning: divide by zero encountered in divide
  division = 1 / data


In [63]:
is_finite_div = np.isfinite(division)
print("Is finite:", is_finite_div)

is_finite_log = np.isfinite(log)
print("Is finite:", is_finite_log)

Is finite: [ True  True False  True  True False  True]
Is finite: [ True  True False False  True False  True]


In [64]:
count_div = is_finite_div.sum()
print("Count of finite values:", count_div)

count_log = is_finite_log.sum()
print("Count of finite values:", count_log)

Count of finite values: 5
Count of finite values: 4


In [66]:
division[~is_finite_div] =  division[is_finite_div].mean()
print("Division:",division)

log[~is_finite_log] = log[is_finite_log].mean()
print("Log:", log)

Division: [ 0.66666667  0.5         0.14047619 -1.          0.28571429  0.14047619
  0.25      ]
Log: [0.40546511 0.69314718 0.9344174  0.9344174  1.25276297 0.9344174
 1.38629436]


In [70]:
original_count = len(data)
print("Original count:", original_count)
valid_count_div = count_div
print("Valid count div:", valid_count_div)
invalid_count = original_count - valid_count_div
print("Invalid count div:", invalid_count)
valid_count_log = count_log
print("Valid count log:", valid_count_log)
invalid_count = original_count - valid_count_log
print("Invalid count log:", invalid_count)

Original count: 7
Valid count div: 5
Invalid count div: 2
Valid count log: 4
Invalid count log: 3
